In [1]:

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================

@dataclass
class DDPMConfig:
    input_dim: int
    context_length: int = 24
    prediction_length: int = 1

    rnn_type: str = "GRU"   # "GRU" or "LSTM"
    rnn_hidden_dim: int = 64
    rnn_layers: int = 2
    rnn_dropout: float = 0.1

    unet_base_channels: int = 32
    unet_depth: int = 3
    time_embed_dim: int = 128

    diffusion_steps: int = 1000
    beta_start: float = 1e-4
    beta_end: float = 2e-2

    lr: float = 1e-3
    batch_size: int = 128
    epochs: int = 100
    grad_clip: float = 1.0
    weight_decay: float = 0.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    scale_condition: float = 1.0


# ============================================================
# Dataset
# ============================================================

class RollingWindowDataset(Dataset):
    """
    Build (context, target) pairs from a multivariate time series.

    Input series shape:
        [T, D]

    Each sample:
        context: [context_length, D]
        target:  [prediction_length, D]
    """
    def __init__(
        self,
        series: np.ndarray | torch.Tensor,
        context_length: int,
        prediction_length: int,
        stride: int = 1,
        normalize: bool = False,
        eps: float = 1e-6,
    ) -> None:
        super().__init__()

        x = torch.as_tensor(series, dtype=torch.float32)
        if x.ndim != 2:
            raise ValueError(f"series must have shape [T, D], got {tuple(x.shape)}")

        self.context_length = context_length
        self.prediction_length = prediction_length
        self.stride = stride
        self.normalize = normalize
        self.eps = eps

        self.mean = x.mean(dim=0, keepdim=True)
        self.std = x.std(dim=0, keepdim=True).clamp_min(eps)

        if normalize:
            x = (x - self.mean) / self.std

        self.series = x
        self.indices = []
        max_start = len(x) - context_length - prediction_length + 1
        for start in range(0, max_start, stride):
            self.indices.append(start)

        if not self.indices:
            raise ValueError("Not enough data to create even one sample.")

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        start = self.indices[idx]
        c0 = start
        c1 = start + self.context_length
        t1 = c1 + self.prediction_length

        context = self.series[c0:c1]      # [Lc, D]
        target = self.series[c1:t1]       # [Lp, D]

        return {
            "context": context,
            "target": target,
        }


# ============================================================
# UNet blocks
# ============================================================

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        device = timesteps.device
        half_dim = self.dim // 2
        if half_dim == 0:
            return timesteps.float().unsqueeze(-1)

        exponent = torch.exp(
            torch.arange(half_dim, device=device, dtype=torch.float32)
            * -(math.log(10000.0) / max(half_dim - 1, 1))
        )
        emb = timesteps.float().unsqueeze(1) * exponent.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class FiLM(nn.Module):
    def __init__(self, channels: int, cond_dim: int) -> None:
        super().__init__()
        self.to_scale_shift = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, channels * 2),
        )

    def forward(self, x: torch.Tensor, cond_vec: torch.Tensor) -> torch.Tensor:
        gamma, beta = self.to_scale_shift(cond_vec).chunk(2, dim=-1)
        return (1.0 + gamma.unsqueeze(-1)) * x + beta.unsqueeze(-1)


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, cond_dim: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm1d(out_channels)
        self.film1 = FiLM(out_channels, cond_dim)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm1d(out_channels)
        self.film2 = FiLM(out_channels, cond_dim)

    def forward(self, x: torch.Tensor, cond_vec: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.film1(x, cond_vec)
        x = F.silu(x)

        x = self.conv2(x)
        x = self.norm2(x)
        x = self.film2(x, cond_vec)
        x = F.silu(x)
        return x


class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, cond_dim: int) -> None:
        super().__init__()
        self.block = ConvBlock(in_channels, out_channels, cond_dim)

    def forward(self, x: torch.Tensor, cond_vec: torch.Tensor) -> torch.Tensor:
        return self.block(x, cond_vec)


class Up(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
        cond_dim: int,
    ) -> None:
        super().__init__()
        self.block = ConvBlock(in_channels + skip_channels, out_channels, cond_dim)

    def forward(
        self,
        x: torch.Tensor,
        skip: torch.Tensor,
        cond_vec: torch.Tensor,
    ) -> torch.Tensor:
        x = torch.cat([skip, x], dim=1)
        return self.block(x, cond_vec)


class UNet1DSameRes(nn.Module):
    """
    Same-resolution 1D U-Net.

    Keeps the repo's basic idea:
      - x shape is [B, prediction_length, target_dim]
      - cond shape is [B, 1, hidden_dim]
      - if hidden_dim != target_dim, project condition length to target_dim
    """
    def __init__(
        self,
        in_channels: int,
        cond_channels: int,
        time_embed_dim: int = 128,
        base_channels: int = 32,
        depth: int = 3,
    ) -> None:
        super().__init__()
        self.cond_channels = cond_channels
        self.len_proj = nn.ModuleDict()

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_embed_dim),
            nn.Linear(time_embed_dim, time_embed_dim * 2),
            nn.SiLU(),
            nn.Linear(time_embed_dim * 2, time_embed_dim),
        )

        total_in = in_channels + cond_channels
        self.inc = ConvBlock(total_in, base_channels, time_embed_dim)

        ch = base_channels
        enc_channels = [ch]
        self.downs = nn.ModuleList()
        for _ in range(depth):
            self.downs.append(Down(ch, ch * 2, time_embed_dim))
            ch *= 2
            enc_channels.append(ch)

        self.bottleneck = ConvBlock(ch, ch, time_embed_dim)

        self.ups = nn.ModuleList()
        for skip_ch in reversed(enc_channels[:-1]):
            self.ups.append(Up(ch, skip_ch, skip_ch, time_embed_dim))
            ch = skip_ch

        self.out = nn.Conv1d(base_channels, in_channels, kernel_size=1)

    def _project_cond_length(self, cond: torch.Tensor, target_len: int) -> torch.Tensor:
        b, c, cond_len = cond.shape
        key = f"{cond_len}->{target_len}"
        if key not in self.len_proj:
            self.len_proj[key] = nn.Linear(cond_len, target_len, bias=False).to(cond.device)
        proj = self.len_proj[key]
        cond = proj(cond.reshape(b * c, cond_len)).reshape(b, c, target_len)
        return cond

    def forward(
        self,
        x: torch.Tensor,              # [B, pred_len, target_dim]
        t: torch.Tensor,              # [B]
        cond: Optional[torch.Tensor], # [B, 1, hidden_dim]
    ) -> torch.Tensor:
        if self.cond_channels > 0:
            if cond is None:
                raise ValueError("cond is required")
            if cond.shape[1] != self.cond_channels:
                raise ValueError(
                    f"Expected cond_channels={self.cond_channels}, got {cond.shape[1]}"
                )
            if cond.shape[-1] != x.shape[-1]:
                cond = self._project_cond_length(cond, x.shape[-1])
            x = x + cond
            x = torch.cat([x, cond], dim=1)

        cond_vec = self.time_mlp(t)

        skips = []
        x = self.inc(x, cond_vec)
        skips.append(x)

        for down in self.downs:
            x = down(x, cond_vec)
            skips.append(x)

        x = self.bottleneck(x, cond_vec)

        for up, skip in zip(self.ups, reversed(skips[:-1])):
            x = up(x, skip, cond_vec)

        return self.out(x)


# ============================================================
# Conditional DDPM
# ============================================================

class ConditionalRecurrentDDPM(nn.Module):
    """
    Recurrent conditioner + 1D U-Net denoiser.

    Training:
        - sample diffusion step t
        - add noise to target x0 -> xt
        - predict epsilon
        - optimize MSE(eps_pred, eps)

    Sampling:
        - DDIM update with configurable eta
        - use eta=1.0 to match your requested setting
    """
    def __init__(self, cfg: DDPMConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.input_dim = cfg.input_dim
        self.target_dim = cfg.input_dim
        self.prediction_length = cfg.prediction_length
        self.feature_dim = cfg.rnn_hidden_dim
        self.scale_condition = cfg.scale_condition

        rnn_cls = {"GRU": nn.GRU, "LSTM": nn.LSTM}[cfg.rnn_type.upper()]
        self.rnn = rnn_cls(
            input_size=cfg.input_dim,
            hidden_size=cfg.rnn_hidden_dim,
            num_layers=cfg.rnn_layers,
            dropout=cfg.rnn_dropout if cfg.rnn_layers > 1 else 0.0,
            batch_first=True,
        )

        self.denoiser = UNet1DSameRes(
            in_channels=cfg.prediction_length,
            cond_channels=1,
            time_embed_dim=cfg.time_embed_dim,
            base_channels=cfg.unet_base_channels,
            depth=cfg.unet_depth,
        )

        betas = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.diffusion_steps, dtype=torch.float32)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        alpha_bars_prev = torch.cat([torch.ones(1), alpha_bars[:-1]], dim=0)

        self.num_steps = cfg.diffusion_steps

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bars", alpha_bars)
        self.register_buffer("alpha_bars_prev", alpha_bars_prev)
        self.register_buffer("sqrt_alpha_bars", torch.sqrt(alpha_bars))
        self.register_buffer("sqrt_one_minus_alpha_bars", torch.sqrt(1.0 - alpha_bars))
        self.register_buffer("sqrt_recip_alpha_bars", torch.sqrt(1.0 / alpha_bars))
        self.register_buffer(
            "sqrt_recipm1_alpha_bars",
            torch.sqrt(torch.clamp(1.0 / alpha_bars - 1.0, min=0.0)),
        )

    def encode_context(self, context: torch.Tensor) -> torch.Tensor:
        """
        context: [B, context_length, input_dim]
        returns cond: [B, 1, hidden_dim]
        """
        _, hidden = self.rnn(context)

        if isinstance(hidden, tuple):  # LSTM
            hidden = hidden[0]

        cond = hidden[-1].unsqueeze(1)  # [B, 1, H]
        cond = cond * self.scale_condition
        return cond

    def predict_eps(
        self,
        x_t: torch.Tensor,
        t: torch.Tensor,
        cond: torch.Tensor,
    ) -> torch.Tensor:
        return self.denoiser(x_t, t, cond)

    def q_sample(
        self,
        x0: torch.Tensor,
        t: torch.Tensor,
        noise: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0)

        sqrt_ab = self.sqrt_alpha_bars[t].view(-1, 1, 1)
        sqrt_omb = self.sqrt_one_minus_alpha_bars[t].view(-1, 1, 1)
        return sqrt_ab * x0 + sqrt_omb * noise

    def training_loss(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        context: [B, Lc, D]
        target:  [B, Lp, D]
        """
        b = context.size(0)
        t = torch.randint(0, self.num_steps, (b,), device=context.device, dtype=torch.long)
        noise = torch.randn_like(target)

        x_t = self.q_sample(target, t, noise)
        cond = self.encode_context(context)
        eps_pred = self.predict_eps(x_t, t, cond)

        loss = F.mse_loss(eps_pred, noise)
        return loss, {"loss": float(loss.detach().item())}

    def forward(
        self,
        context: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        loss, _ = self.training_loss(context, target)
        return loss

    @torch.no_grad()
    def ddim_sample(
        self,
        context: torch.Tensor,
        num_samples: int = 1,
        eta: float = 1.0,
        clip_x0: Optional[float] = None,
    ) -> torch.Tensor:
        """
        context:
            [B, context_length, D]

        returns:
            [B, num_samples, prediction_length, D]
        """
        self.eval()
        device = context.device
        batch_size = context.size(0)

        cond = self.encode_context(context)                      # [B, 1, H]
        cond = cond.repeat_interleave(num_samples, dim=0)       # [B*num_samples, 1, H]

        x = torch.randn(
            batch_size * num_samples,
            self.prediction_length,
            self.target_dim,
            device=device,
        )

        for step in reversed(range(self.num_steps)):
            t = torch.full((x.size(0),), step, device=device, dtype=torch.long)
            eps = self.predict_eps(x, t, cond)

            alpha_bar_t = self.alpha_bars[step]
            alpha_bar_prev = self.alpha_bars_prev[step]

            sqrt_alpha_bar_t = torch.sqrt(alpha_bar_t)
            sqrt_one_minus_alpha_bar_t = torch.sqrt(1.0 - alpha_bar_t)

            x0_pred = (x - sqrt_one_minus_alpha_bar_t * eps) / sqrt_alpha_bar_t

            if clip_x0 is not None:
                x0_pred = x0_pred.clamp(-clip_x0, clip_x0)

            if step == 0:
                x = x0_pred
                continue

            sigma_t = eta * torch.sqrt(
                ((1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t))
                * (1.0 - alpha_bar_t / alpha_bar_prev)
            )
            noise = torch.randn_like(x)
            coeff_eps = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma_t ** 2, min=0.0))

            x = (
                torch.sqrt(alpha_bar_prev) * x0_pred
                + coeff_eps * eps
                + sigma_t * noise
            )

        x = x.view(batch_size, num_samples, self.prediction_length, self.target_dim)
        return x

    @torch.no_grad()
    def sample_autoregressive(
        self,
        history: torch.Tensor,
        horizon: int,
        num_samples: int = 1,
        eta: float = 1.0,
        clip_x0: Optional[float] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Roll forward one step at a time, like the repo's offline scenario generation.

        history:
            [context_length, D]

        returns:
            sampled_returns:  [horizon, num_samples, D]
            sampled_features: [horizon, num_samples, H]
        """
        self.eval()
        device = next(self.parameters()).device

        if history.ndim != 2:
            raise ValueError(f"history must have shape [context_length, D], got {tuple(history.shape)}")
        if history.shape[0] != self.cfg.context_length:
            raise ValueError(
                f"history length must equal context_length={self.cfg.context_length}, "
                f"got {history.shape[0]}"
            )

        context = history.unsqueeze(0).repeat(num_samples, 1, 1).to(device)
        all_returns = []
        all_features = []

        for _ in range(horizon):
            cond = self.encode_context(context)                     # [num_samples, 1, H]
            all_features.append(cond[:, 0, :].detach().cpu())

            block = self.ddim_sample(
                context=context,
                num_samples=1,
                eta=eta,
                clip_x0=clip_x0,
            )[:, 0]                                                # [num_samples, pred_len, D]

            next_return = block[:, 0, :]                           # [num_samples, D]
            all_returns.append(next_return.detach().cpu())

            context = torch.cat([context[:, 1:, :], next_return.unsqueeze(1)], dim=1)

        sampled_returns = torch.stack(all_returns, dim=0)
        sampled_features = torch.stack(all_features, dim=0)
        return sampled_returns, sampled_features


# ============================================================
# Training / inference helpers
# ============================================================

def build_dataloader(
    series: np.ndarray | torch.Tensor,
    cfg: DDPMConfig,
    shuffle: bool = True,
    stride: int = 1,
    normalize: bool = False,
    num_workers: int = 0,
    drop_last: bool = True,
) -> tuple[RollingWindowDataset, DataLoader]:
    dataset = RollingWindowDataset(
        series=series,
        context_length=cfg.context_length,
        prediction_length=cfg.prediction_length,
        stride=stride,
        normalize=normalize,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
        pin_memory=torch.cuda.is_available(),
    )
    return dataset, loader


def train_model(
    model: ConditionalRecurrentDDPM,
    train_loader: DataLoader,
    cfg: DDPMConfig,
    valid_loader: Optional[DataLoader] = None,
) -> Dict[str, list[float]]:
    device = torch.device(cfg.device)
    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    history = {"train_loss": [], "valid_loss": []}

    for epoch in range(cfg.epochs):
        model.train()
        running = 0.0
        n_batches = 0

        for batch in train_loader:
            context = batch["context"].to(device)   # [B, Lc, D]
            target = batch["target"].to(device)     # [B, Lp, D]

            optimizer.zero_grad(set_to_none=True)
            loss, _ = model.training_loss(context, target)
            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

            optimizer.step()

            running += loss.item()
            n_batches += 1

        train_loss = running / max(n_batches, 1)
        history["train_loss"].append(train_loss)

        if valid_loader is not None:
            model.eval()
            val_running = 0.0
            val_batches = 0
            with torch.no_grad():
                for batch in valid_loader:
                    context = batch["context"].to(device)
                    target = batch["target"].to(device)
                    loss, _ = model.training_loss(context, target)
                    val_running += loss.item()
                    val_batches += 1
            valid_loss = val_running / max(val_batches, 1)
            history["valid_loss"].append(valid_loss)
            print(
                f"Epoch {epoch + 1:03d}/{cfg.epochs:03d} | "
                f"train={train_loss:.6f} | valid={valid_loss:.6f}"
            )
        else:
            print(f"Epoch {epoch + 1:03d}/{cfg.epochs:03d} | train={train_loss:.6f}")

    return history


@torch.no_grad()
def generate_scenarios(
    model: ConditionalRecurrentDDPM,
    history_window: np.ndarray | torch.Tensor,
    horizon: int,
    num_samples: int,
    eta: float = 1.0,
    clip_x0: Optional[float] = None,
) -> Dict[str, torch.Tensor]:
    """
    Returns the same kind of objects the original repo conceptually wants:
      - sampled returns
      - sampled price paths built from returns
      - latent features from the recurrent encoder
    """
    device = next(model.parameters()).device
    history = torch.as_tensor(history_window, dtype=torch.float32, device=device)

    sampled_returns, sampled_features = model.sample_autoregressive(
        history=history,
        horizon=horizon,
        num_samples=num_samples,
        eta=eta,
        clip_x0=clip_x0,
    )

    horizon_steps, sample_count, dim = sampled_returns.shape
    price_paths = torch.ones(horizon_steps + 1, sample_count, dim)

    for t in range(horizon_steps):
        price_paths[t + 1] = price_paths[t] * (1.0 + sampled_returns[t])

    return {
        "returns": sampled_returns,
        "prices": price_paths,
        "features": sampled_features,
    }


# ============================================================
# Example
# ============================================================



In [2]:

    # Fake multivariate return series: [T, D]
T = 2000
D = 8
rng = np.random.default_rng(42)
series = rng.normal(0.001, 0.02, size=(T, D)).astype(np.float32)

cfg = DDPMConfig(
    input_dim=D,
    context_length=24,
    prediction_length=1,
    rnn_type="GRU",
    rnn_hidden_dim=64,
    rnn_layers=2,
    unet_base_channels=32,
    unet_depth=3,
    diffusion_steps=200,
    beta_start=1e-4,
    beta_end=2e-2,
    lr=1e-3,
    batch_size=64,
    epochs=5,
)

dataset, train_loader = build_dataloader(
    series=series,
    cfg=cfg,
    shuffle=True,
    stride=1,
    normalize=True,
)

model = ConditionalRecurrentDDPM(cfg)
train_model(model, train_loader, cfg)

history_window = series[-cfg.context_length:]  # [context_length, D]
out = generate_scenarios(
    model=model.to(cfg.device),
    history_window=history_window,
    horizon=12,
    num_samples=32,
    eta=1.0,   # requested DDIM setting
    clip_x0=None,
)

print("returns:", tuple(out["returns"].shape))   # [H, S, D]
print("prices:", tuple(out["prices"].shape))     # [H+1, S, D]
print("features:", tuple(out["features"].shape)) # [H, S, Hdim]


Epoch 001/005 | train=0.683477
Epoch 002/005 | train=0.609693
Epoch 003/005 | train=0.607610
Epoch 004/005 | train=0.599048
Epoch 005/005 | train=0.590703


KeyboardInterrupt: 

In [ ]:
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Dict, List

import numpy as np
import matplotlib.pyplot as plt
import torch
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning


# ============================================================
# Synthetic ARMA RETURN data generation
# ============================================================

def simulate_multivariate_arma(
    T: int,
    D: int,
    ar_params: List[float],
    ma_params: List[float],
    mu: float = 0.0,
    noise_scale: float = 0.05,
    burnin: int = 500,
    cross_corr: float = 0.25,
    seed: int = 42,
) -> np.ndarray:
    """
    Simulate a multivariate ARMA(p, q) RETURN process with correlated Gaussian shocks.

    r_t[d] = mu + sum_i phi_i r_{t-i}[d] + e_t[d] + sum_j theta_j e_{t-j}[d]

    Each dimension shares the same AR/MA coefficients, but innovations are
    correlated across dimensions.

    Returns:
        returns: [T, D]
    """
    rng = np.random.default_rng(seed)

    p = len(ar_params)
    q = len(ma_params)

    total = T + burnin

    x = np.zeros((total, D), dtype=np.float64)
    e = np.zeros((total, D), dtype=np.float64)

    cov = np.full((D, D), cross_corr, dtype=np.float64)
    np.fill_diagonal(cov, 1.0)
    cov = (noise_scale ** 2) * cov

    shocks = rng.multivariate_normal(
        mean=np.zeros(D, dtype=np.float64),
        cov=cov,
        size=total,
    )

    start = max(p, q)

    for t in range(start, total):
        ar_term = np.zeros(D, dtype=np.float64)
        ma_term = np.zeros(D, dtype=np.float64)

        for i, phi in enumerate(ar_params, start=1):
            ar_term += phi * x[t - i]

        for j, theta in enumerate(ma_params, start=1):
            ma_term += theta * e[t - j]

        e[t] = shocks[t]
        x[t] = mu + ar_term + e[t] + ma_term

    return x[burnin:].astype(np.float32)


# ============================================================
# Price / return helpers
# ============================================================

def prices_to_log_returns(prices: np.ndarray) -> np.ndarray:
    """
    Convert prices to log returns.

    prices: [T] or [T, D]
    returns: [T-1, D]
    """
    prices = np.asarray(prices, dtype=np.float32)

    if prices.ndim == 1:
        prices = prices[:, None]

    if np.any(~np.isfinite(prices)):
        raise ValueError("prices contain NaN or infinite values.")

    if np.any(prices <= 0):
        raise ValueError("prices must be strictly positive to compute log returns.")

    log_prices = np.log(prices)
    returns = np.diff(log_prices, axis=0)

    return returns.astype(np.float32)


def returns_to_prices(
    returns: np.ndarray,
    initial_price: np.ndarray | float,
) -> np.ndarray:
    """
    Convert log returns back to prices.

    returns: [T, D]
    initial_price: scalar or [D]
    prices: [T, D]
    """
    returns = np.asarray(returns, dtype=np.float32)

    if returns.ndim == 1:
        returns = returns[:, None]

    initial_price = np.asarray(initial_price, dtype=np.float32)

    if initial_price.ndim == 0:
        initial_price = np.full(returns.shape[1], float(initial_price), dtype=np.float32)

    if np.any(initial_price <= 0):
        raise ValueError("initial_price must be strictly positive.")

    log_prices = np.log(initial_price)[None, :] + np.cumsum(returns, axis=0)
    prices = np.exp(log_prices)

    return prices.astype(np.float32)


def make_synthetic_prices_from_arma_returns(
    T: int = 3000,
    D: int = 4,
    initial_price: float = 100.0,
    true_ar: List[float] = [0.65, -0.20],
    true_ma: List[float] = [0.45],
    true_mu: float = 0.001,
    true_sigma: float = 0.03,
    cross_corr: float = 0.30,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, object]]:
    """
    Create synthetic ARMA returns, then convert them into synthetic prices.
    This mimics real finance data:

        ARMA returns -> prices -> log returns again for training
    """
    synthetic_returns = simulate_multivariate_arma(
        T=T,
        D=D,
        ar_params=true_ar,
        ma_params=true_ma,
        mu=true_mu,
        noise_scale=true_sigma,
        burnin=800,
        cross_corr=cross_corr,
        seed=seed,
    )

    initial_price_vec = initial_price * np.ones(D, dtype=np.float32)

    synthetic_prices = returns_to_prices(
        returns=synthetic_returns,
        initial_price=initial_price_vec,
    )

    info = {
        "true_ar": true_ar,
        "true_ma": true_ma,
        "true_mu": true_mu,
        "true_sigma": true_sigma,
        "cross_corr": cross_corr,
        "seed": seed,
    }

    return synthetic_prices, synthetic_returns, info


# ============================================================
# Diagnostics
# ============================================================

def autocorr(x: np.ndarray, max_lag: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]

    x = x - x.mean()

    denom = np.dot(x, x)

    if denom <= 1e-12:
        return np.zeros(max_lag + 1, dtype=np.float64)

    acf = [1.0]

    for lag in range(1, max_lag + 1):
        num = np.dot(x[:-lag], x[lag:])
        acf.append(num / denom)

    return np.asarray(acf, dtype=np.float64)


def fit_best_arma(
    x: np.ndarray,
    max_p: int = 3,
    max_q: int = 3,
    verbose: bool = True,
) -> Dict[str, object]:
    """
    Fit a grid of ARMA(p, q) models using ARIMA(p,0,q) and keep the lowest AIC.
    ARMA is fitted to RETURNS, not prices.
    """
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]

    if len(x) < 50:
        raise ValueError(f"Series is too short for ARMA fitting. Got length={len(x)}.")

    if np.std(x) <= 1e-12:
        raise ValueError("Series is almost constant; ARMA fitting is not meaningful.")

    best = None

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    warnings.simplefilter("ignore", UserWarning)
                    res = ARIMA(x, order=(p, 0, q), trend="c").fit()

                item = {
                    "order": (p, q),
                    "aic": float(res.aic),
                    "bic": float(res.bic),
                    "params": dict(zip(res.param_names, res.params)),
                    "result": res,
                }

                if best is None or item["aic"] < best["aic"]:
                    best = item

            except Exception as e:
                if verbose:
                    print(f"Failed ARMA({p},{q}): {e}")
                continue

    if best is None:
        raise RuntimeError("ARMA fitting failed for all candidate orders.")

    return best


def summarize_process(
    x: np.ndarray,
    name: str,
    max_lag: int = 20,
    max_p: int = 3,
    max_q: int = 3,
) -> Dict[str, object]:
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]

    acf = autocorr(x, max_lag=max_lag)
    best = fit_best_arma(x, max_p=max_p, max_q=max_q)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"length={len(x)}")
    print(f"mean={x.mean():.6f}, std={x.std():.6f}")
    print(f"best ARMA order by AIC: {best['order']}, AIC={best['aic']:.2f}, BIC={best['bic']:.2f}")
    print("params:")

    for k, v in best["params"].items():
        print(f"  {k}: {v:.6f}")

    print(f"acf[1:{max_lag+1}] = {np.round(acf[1:], 4)}")

    return {
        "acf": acf,
        "best": best,
    }


# ============================================================
# Normalization helpers
# ============================================================

def _to_numpy(x) -> np.ndarray:
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def get_dataset_normalization_stats(
    dataset,
    raw_series: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Try to get mean/std from the dataset.
    If unavailable, compute mean/std from raw returns.

    This is used because build_dataloader(..., normalize=True) trains
    the diffusion model on normalized returns.
    """
    raw_series = np.asarray(raw_series, dtype=np.float32)

    if hasattr(dataset, "mean") and hasattr(dataset, "std"):
        mean = _to_numpy(dataset.mean).astype(np.float32)
        std = _to_numpy(dataset.std).astype(np.float32)

    elif hasattr(dataset, "mu") and hasattr(dataset, "sigma"):
        mean = _to_numpy(dataset.mu).astype(np.float32)
        std = _to_numpy(dataset.sigma).astype(np.float32)

    else:
        mean = raw_series.mean(axis=0, keepdims=True).astype(np.float32)
        std = raw_series.std(axis=0, keepdims=True).astype(np.float32)

    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)

    if mean.ndim == 1:
        mean = mean[None, :]

    if std.ndim == 1:
        std = std[None, :]

    std = np.maximum(std, 1e-8).astype(np.float32)

    return mean, std


# ============================================================
# Sampling helper
# ============================================================

@torch.no_grad()
def sample_long_trajectory(
    model: ConditionalRecurrentDDPM,
    seed_history: np.ndarray | torch.Tensor,
    horizon: int,
    eta: float = 0.0,
    clip_x0: Optional[float] = 5.0,
) -> np.ndarray:
    """
    Generate one long multivariate trajectory autoregressively.

    Important:
    Here the generated trajectory is in the same scale as the model input.
    Since the model is trained with normalize=True, this function returns
    normalized returns. We inverse-normalize after sampling.

    Returns:
        generated_norm: [horizon, D]
    """
    model.eval()

    device = next(model.parameters()).device

    context = torch.as_tensor(
        seed_history,
        dtype=torch.float32,
        device=device,
    ).unsqueeze(0)

    if context.shape[1] != model.cfg.context_length:
        raise ValueError(
            f"seed_history length must be {model.cfg.context_length}, got {context.shape[1]}"
        )

    out = []

    for _ in range(horizon):
        block = model.ddim_sample(
            context=context,
            num_samples=1,
            eta=eta,
            clip_x0=clip_x0,
        )[:, 0]

        next_x = block[:, 0, :]

        out.append(next_x[0].detach().cpu().numpy())

        context = torch.cat(
            [context[:, 1:, :], next_x.unsqueeze(1)],
            dim=1,
        )

    return np.asarray(out, dtype=np.float32)


# ============================================================
# Main experiment
# ============================================================

def run_arma_diffusion_experiment(
    prices: Optional[np.ndarray] = None,
    horizon: int = 500,
    dim: int = 0,
) -> None:
    # --------------------------------------------------------
    # 1) Get prices, then convert prices to log returns
    # --------------------------------------------------------
    #
    # If you have real prices:
    #
    #     run_arma_diffusion_experiment(prices=your_prices)
    #
    # If prices=None, this function creates synthetic ARMA returns,
    # converts them into synthetic prices, then converts prices back
    # into returns. This keeps the workflow the same as real finance.
    # --------------------------------------------------------

    synthetic_info = None

    if prices is None:
        prices, synthetic_returns, synthetic_info = make_synthetic_prices_from_arma_returns(
            T=3000,
            D=4,
            initial_price=100.0,
            true_ar=[0.65, -0.20],
            true_ma=[0.45],
            true_mu=0.001,
            true_sigma=0.03,
            cross_corr=0.30,
            seed=42,
        )

        print("Using synthetic prices generated from ARMA returns.")

    else:
        prices = np.asarray(prices, dtype=np.float32)

        if prices.ndim == 1:
            prices = prices[:, None]

        print("Using user-provided price data.")

    prices = np.asarray(prices, dtype=np.float32)

    if prices.ndim == 1:
        prices = prices[:, None]

    if np.any(~np.isfinite(prices)):
        raise ValueError("prices contain NaN or infinite values.")

    if np.any(prices <= 0):
        raise ValueError("prices must be strictly positive to compute log returns.")

    # This is the important change:
    # train diffusion on log returns, not raw prices.
    returns = prices_to_log_returns(prices)

    series = returns

    T_returns, D = series.shape

    if dim < 0 or dim >= D:
        raise ValueError(f"dim must be between 0 and {D - 1}, got dim={dim}")

    print("\nData")
    print("----")
    print("prices shape:", prices.shape)
    print("returns shape:", series.shape)
    print("returns mean:", np.round(series.mean(axis=0), 6))
    print("returns std:", np.round(series.std(axis=0), 6))

    # --------------------------------------------------------
    # 2) Build DataLoader on returns
    # --------------------------------------------------------

    cfg = DDPMConfig(
        input_dim=D,
        context_length=64,
        prediction_length=1,
        rnn_type="GRU",
        rnn_hidden_dim=64,
        rnn_layers=2,
        unet_base_channels=16,
        unet_depth=2,
        diffusion_steps=50,
        beta_start=1e-4,
        beta_end=2e-2,
        lr=1e-3,
        batch_size=128,
        epochs=30,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    dataset, train_loader = build_dataloader(
        series=series,
        cfg=cfg,
        shuffle=True,
        stride=1,
        normalize=True,
    )

    mean, std = get_dataset_normalization_stats(
        dataset=dataset,
        raw_series=series,
    )

    print("\nNormalization")
    print("-------------")
    print("mean shape:", mean.shape)
    print("std shape:", std.shape)

    # --------------------------------------------------------
    # 3) Train diffusion model on normalized returns
    # --------------------------------------------------------

    model = ConditionalRecurrentDDPM(cfg)

    train_model(
        model=model,
        train_loader=train_loader,
        cfg=cfg,
    )

    model = model.to(cfg.device)

    # --------------------------------------------------------
    # 4) Generate normalized returns autoregressively
    # --------------------------------------------------------
    #
    # Because normalize=True, the model expects normalized history.
    # So we use normalized return history as seed.
    # --------------------------------------------------------

    raw_history = series[-cfg.context_length:]

    history_window_norm = (raw_history - mean) / std

    generated_norm = sample_long_trajectory(
        model=model,
        seed_history=history_window_norm,
        horizon=horizon,
        eta=0.0,
        clip_x0=5.0,
    )

    # Inverse normalization:
    # generated normalized returns -> generated real returns
    generated_returns = generated_norm * std + mean

    generated_returns = generated_returns.astype(np.float32)

    print("\nGeneration")
    print("----------")
    print("generated normalized returns shape:", generated_norm.shape)
    print("generated returns shape:", generated_returns.shape)

    # --------------------------------------------------------
    # 5) Fit ARMA on returns, not prices
    # --------------------------------------------------------

    real_1d = series[:, dim]
    gen_1d = generated_returns[:, dim]

    real_stats = summarize_process(
        real_1d,
        name=f"Original real returns (dim={dim})",
    )

    gen_stats = summarize_process(
        gen_1d,
        name=f"Generated diffusion returns (dim={dim})",
    )

    # --------------------------------------------------------
    # 6) Reconstruct generated prices only for visualization
    # --------------------------------------------------------
    #
    # Do NOT fit ARMA on generated prices.
    # ARMA diagnostics belong on generated returns.
    # --------------------------------------------------------

    last_price = prices[-1]

    generated_prices = returns_to_prices(
        returns=generated_returns,
        initial_price=last_price,
    )

    print("\nPrice reconstruction")
    print("--------------------")
    print("generated prices shape:", generated_prices.shape)

    # --------------------------------------------------------
    # 7) Plot comparisons
    # --------------------------------------------------------

    max_lag = 20
    lags = np.arange(max_lag + 1)

    fig = plt.figure(figsize=(14, 10))

    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(real_1d[:300], label="real returns")
    ax1.set_title("Original returns, first 300")
    ax1.legend()

    ax2 = fig.add_subplot(3, 2, 2)
    ax2.plot(gen_1d[:300], label="generated returns")
    ax2.set_title("Generated returns, first 300")
    ax2.legend()

    ax3 = fig.add_subplot(3, 2, 3)
    ax3.stem(lags, real_stats["acf"], label="real returns", basefmt=" ")
    ax3.stem(lags + 0.05, gen_stats["acf"], label="generated returns", basefmt=" ")
    ax3.set_title("ACF comparison on returns")
    ax3.set_xlabel("lag")
    ax3.legend()

    ax4 = fig.add_subplot(3, 2, 4)
    ax4.hist(real_1d, bins=40, alpha=0.6, density=True, label="real returns")
    ax4.hist(gen_1d, bins=40, alpha=0.6, density=True, label="generated returns")
    ax4.set_title("Return distribution")
    ax4.legend()

    ax5 = fig.add_subplot(3, 2, 5)
    ax5.plot(prices[-300:, dim], label="real price")
    ax5.set_title("Original price, last 300")
    ax5.legend()

    ax6 = fig.add_subplot(3, 2, 6)
    ax6.plot(generated_prices[:300, dim], label="generated price")
    ax6.set_title("Generated price reconstructed from generated returns")
    ax6.legend()

    fig.tight_layout()

    out_dir = Path("/mnt/data")
    fig_path = out_dir / "return_diffusion_arma_comparison.png"

    fig.savefig(
        fig_path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"\nSaved plot to: {fig_path}")

    # --------------------------------------------------------
    # 8) Print interpretation
    # --------------------------------------------------------

    if synthetic_info is not None:
        print("\nGround-truth synthetic return simulator")
        print("---------------------------------------")
        print(f"True AR params: {synthetic_info['true_ar']}")
        print(f"True MA params: {synthetic_info['true_ma']}")
        print(f"True return mean: {synthetic_info['true_mu']}")
        print(f"True innovation std: {synthetic_info['true_sigma']}")

    print("\nInterpretation")
    print("--------------")
    print(
        "The diffusion model is trained on log returns, not raw prices. "
        "The ARMA diagnostics are also applied to returns. "
        "Generated prices are reconstructed only after sampling, for visualization. "
        "If the diffusion model captures the return dynamics, the generated returns "
        "should have a similar ACF shape and a similar low-order ARMA fit."
    )


# ============================================================
# Run
# ============================================================
# For synthetic test:
run_arma_diffusion_experiment()

# For your own real price data, comment the line above and use:
# run_arma_diffusion_experiment(prices=your_prices)

In [ ]:
run_arma_diffusion_experiment()

series shape: (3000, 4)
Epoch 001/020 | train=0.243904
Epoch 002/020 | train=0.065214
Epoch 003/020 | train=0.067362
Epoch 004/020 | train=0.059772
Epoch 005/020 | train=0.053974
Epoch 006/020 | train=0.047912
Epoch 007/020 | train=0.051462
Epoch 008/020 | train=0.056411
Epoch 009/020 | train=0.049065
Epoch 010/020 | train=0.051712
Epoch 011/020 | train=0.047497
Epoch 012/020 | train=0.049296
Epoch 013/020 | train=0.043989
Epoch 014/020 | train=0.042894
Epoch 015/020 | train=0.039745
Epoch 016/020 | train=0.040017
Epoch 017/020 | train=0.046524
Epoch 018/020 | train=0.042169
Epoch 019/020 | train=0.042915
Epoch 020/020 | train=0.037367


KeyboardInterrupt: 

In [ ]:
T = 3000
D = 4

# True process: ARMA(2,1)
true_ar = [0.65, -0.20]
true_ma = [0.45]
true_mu = 0.001
true_sigma = 0.03

series = simulate_multivariate_arma(
  T=T,
  D=D,
  ar_params=true_ar,
  ma_params=true_ma,
  mu=true_mu,
  noise_scale=true_sigma,
  burnin=800,
  cross_corr=0.30,
  seed=42,
)

print("series shape:", series.shape)

# --------------------------------------------------------
# 2) Build clean DataLoader
# --------------------------------------------------------
cfg = DDPMConfig(
  input_dim=D,
  context_length=32,
  prediction_length=1,
  rnn_type="GRU",
  rnn_hidden_dim=64,
  rnn_layers=2,
  unet_base_channels=32,
  unet_depth=3,
  diffusion_steps=200,
  beta_start=1e-4,
  beta_end=2e-2,
  lr=1e-3,
  batch_size=128,
  epochs=50,
  device="cuda" if torch.cuda.is_available() else "cpu",
)

dataset, train_loader = build_dataloader(
  series=series,
  cfg=cfg,
  shuffle=True,
  stride=1,
  normalize=False,
)

# --------------------------------------------------------
# 3) Train diffusion model
# --------------------------------------------------------
model = ConditionalRecurrentDDPM(cfg)
train_model(model, train_loader, cfg)
model = model.to(cfg.device)

# --------------------------------------------------------
# 4) Generate one long synthetic trajectory from the model
# --------------------------------------------------------
history_window = series[-cfg.context_length:]     # [context_length, D]
generated = sample_long_trajectory(
  model=model,
  seed_history=history_window,
  horizon=10,
  eta=1.0,
  clip_x0=None,
)

print("generated shape:", generated.shape)



series shape: (3000, 4)
Epoch 001/050 | train=0.255548
Epoch 002/050 | train=0.072375
Epoch 003/050 | train=0.067401
Epoch 004/050 | train=0.050938
Epoch 005/050 | train=0.054107
Epoch 006/050 | train=0.050402
Epoch 007/050 | train=0.052059
Epoch 008/050 | train=0.053292
Epoch 009/050 | train=0.058352
Epoch 010/050 | train=0.042781
Epoch 011/050 | train=0.049238
Epoch 012/050 | train=0.047068
Epoch 013/050 | train=0.049015
Epoch 014/050 | train=0.054957
Epoch 015/050 | train=0.039308
Epoch 016/050 | train=0.039575
Epoch 017/050 | train=0.046049
Epoch 018/050 | train=0.043816
Epoch 019/050 | train=0.042477
Epoch 020/050 | train=0.041843
Epoch 021/050 | train=0.040121
Epoch 022/050 | train=0.036672
Epoch 023/050 | train=0.032910
Epoch 024/050 | train=0.042215
Epoch 025/050 | train=0.038536
Epoch 026/050 | train=0.036064
Epoch 027/050 | train=0.040152
Epoch 028/050 | train=0.034886
Epoch 029/050 | train=0.036399
Epoch 030/050 | train=0.040139
Epoch 031/050 | train=0.034926
Epoch 032/050 |